In [0]:
"""
Data Quality - Accuracy Check Task
Evaluates structured business rules and stores 1 / 0 / NULL results.
"""
from pyspark.sql import functions as F
import json
import re


# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("custom_rules_json", "[]")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
CUSTOM_RULES_JSON = dbutils.widgets.get("custom_rules_json")


def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', str(name)).lower()


def _parse_rules_payload(payload):
    """Accept either a list of rules, a dict with custom_rules, or the legacy format."""
    if not payload:
        return []

    try:
        parsed = json.loads(payload)
    except Exception as exc:
        raise ValueError(f"Invalid custom_rules_json payload: {exc}")

    if isinstance(parsed, list):
        return parsed

    if isinstance(parsed, dict):
        if isinstance(parsed.get('custom_rules'), list):
            return parsed['custom_rules']
        if isinstance(parsed.get('controls'), list):
            return parsed['controls']

    raise ValueError("custom_rules_json must decode to a list or an object containing 'custom_rules'")


def _sql_column_name(column_name):
    if not column_name:
        return ''
    return f"`{str(column_name).replace('`', '``')}`"


def _resolve_column_name(column_name, actual_columns):
    """Resolve a user-provided column name against the prepared input schema."""
    if not column_name:
        return ''

    if column_name in actual_columns:
        return actual_columns[column_name]

    return actual_columns.get(str(column_name).lower(), column_name)


def _format_literal(value):
    if value is None:
        return 'NULL'

    text = str(value).strip()
    if text == '':
        return "''"
    if re.fullmatch(r'-?\d+(\.\d+)?', text):
        return text
    if text.lower() in ('true', 'false'):
        return text.lower()

    return "'" + text.replace("'", "''") + "'"


def _literal_matches_type(value, value_type):
    if value is None:
        return False

    text = str(value).strip()
    if text == '':
        return False

    value_type = (value_type or 'text').strip().lower()

    if value_type == 'numeric':
        return bool(re.fullmatch(r'-?\d+(\.\d+)?', text))
    if value_type == 'date':
        return bool(
            re.fullmatch(r'\d{2}-\d{2}-\d{4}', text)
            or re.fullmatch(r'\d{4}-\d{2}-\d{2}', text)
            or re.fullmatch(r'\d{2}/\d{2}/\d{4}', text)
        )

    return True


def _format_typed_literal(value, value_type):
    value_type = (value_type or 'text').strip().lower()
    text = '' if value is None else str(value).strip()

    if value_type == 'numeric' and _literal_matches_type(text, 'numeric'):
        return text

    return _format_literal(text)


def _literal_type_warning(value, value_type):
    if value is None:
        return None

    text = str(value).strip()
    if text == '':
        return None

    value_type = (value_type or 'text').strip().lower()
    if value_type not in ('numeric', 'text', 'date'):
        value_type = 'text'

    if _literal_matches_type(text, value_type):
        return None

    if value_type == 'numeric':
        return 'Warning: numeric selected, but the value does not look numeric.'
    if value_type == 'date':
        return 'Warning: date selected, but the value does not look like a date (DD-MM-YYYY, YYYY-MM-DD, or DD/MM/YYYY).'

    return None


def _missing_expression(columns):
    missing_expr = None
    for column_name in columns:
        expr = F.col(column_name).isNull() | (F.trim(F.col(column_name).cast('string')) == '')
        missing_expr = expr if missing_expr is None else (missing_expr | expr)
    return missing_expr


def _condition_sql(condition, actual_columns):
    """Build a Spark SQL fragment from a structured condition."""
    if not isinstance(condition, dict):
        return ''

    left_column = _resolve_column_name(condition.get('left_column', ''), actual_columns)
    operator = (condition.get('operator', '==') or '==').strip().lower()
    right_mode = (condition.get('right_mode', 'column') or 'column').strip().lower()
    value_type = condition.get('right_value_type', 'text')

    if not left_column:
        return ''

    left_expr = _sql_column_name(left_column)

    if operator in ('is null', 'is not null'):
        return f"{left_expr} {operator.upper()}"

    if right_mode == 'column':
        right_ref = _resolve_column_name(condition.get('right_column', ''), actual_columns)
        if not right_ref:
            return ''
        right_expr = _sql_column_name(right_ref)
    else:
        right_expr = _format_typed_literal(condition.get('right_value', ''), value_type)

    if operator in ('==', '='):
        return f"{left_expr} = {right_expr}"
    if operator == '!=':
        return f"{left_expr} != {right_expr}"
    if operator in ('>', '>=', '<', '<='):
        return f"{left_expr} {operator} {right_expr}"
    if operator == 'contains':
        return f"instr(cast({left_expr} as string), cast({right_expr} as string)) > 0"
    if operator == 'not contains':
        return f"instr(cast({left_expr} as string), cast({right_expr} as string)) = 0"

    return ''


def _rule_sql(rule, actual_columns):
    """Translate a structured rule into Spark SQL."""
    if not isinstance(rule, dict):
        return ''

    if rule.get('condition') and not rule.get('conditions'):
        return _rule_sql_legacy(rule.get('condition'))

    conditions = rule.get('conditions', [])
    if not isinstance(conditions, list) or not conditions:
        return _rule_sql_legacy(rule.get('condition', ''))

    expression_parts = []
    for idx, condition in enumerate(conditions):
        fragment = _condition_sql(condition, actual_columns)
        if not fragment:
            continue

        if idx == 0 or not expression_parts:
            expression_parts.append(f"({fragment})")
        else:
            connector = (condition.get('connector') or 'AND').strip().upper()
            if connector not in ('AND', 'OR'):
                connector = 'AND'
            expression_parts.append(f"{connector} ({fragment})")

    return ' '.join(expression_parts)


def _rule_sql_legacy(condition):
    """Translate the legacy Dataiku-like syntax used in old custom rules."""
    if not condition:
        return ""

    expr = condition.strip()
    expr = expr.replace("&&", " AND ")
    expr = expr.replace("||", " OR ")
    expr = re.sub(r'!\s*\(', 'NOT (', expr)
    expr = expr.replace('"', "'")

    expr = re.sub(
        r'arrayContains\s*\(\s*\[(.*?)\]\s*,\s*([^)]+?)\s*\)',
        r'array_contains(array(\1), \2)',
        expr,
        flags=re.IGNORECASE,
    )

    expr = re.sub(
        r'abs\s*\(\s*diff\s*\(\s*([^,]+?)\s*,\s*([^,]+?)\s*,\s*[\'\"]day[\'\"]\s*\)\s*\)',
        r'abs(datediff(\2, \1))',
        expr,
        flags=re.IGNORECASE,
    )

    return expr


def _normalize_rule(rule, idx):
    """Normalize one incoming rule to a consistent backend format."""
    if not isinstance(rule, dict):
        return None

    normalized = dict(rule)
    normalized['name'] = normalized.get('name') or f'Rule {idx}'
    normalized['enabled'] = normalized.get('enabled', True)
    normalized['blank_when_not_applicable'] = normalized.get('blank_when_not_applicable', True)

    # Ensure conditions list exists for the new front contract.
    if not normalized.get('conditions') and any(
        key in normalized for key in ('left_column', 'operator', 'right_mode', 'right_column', 'right_value')
    ):
        normalized['conditions'] = [{
            'id': normalized.get('id', f'rule_{idx}'),
            'connector': '',
            'left_column': normalized.get('left_column', ''),
            'operator': normalized.get('operator', '=='),
            'right_mode': normalized.get('right_mode', 'column'),
            'right_value_type': normalized.get('right_value_type', 'text'),
            'right_column': normalized.get('right_column', ''),
            'right_value': normalized.get('right_value', ''),
        }]

    return normalized


def _rule_referenced_columns(rule, actual_columns):
    """Return the list of resolved columns referenced by a rule."""
    referenced = []

    for condition in rule.get('conditions', []) or []:
        if not isinstance(condition, dict):
            continue

        left_column = _resolve_column_name(condition.get('left_column', ''), actual_columns)
        if left_column:
            referenced.append(left_column)

        if (condition.get('right_mode', 'column') or 'column').strip().lower() == 'column':
            right_column = _resolve_column_name(condition.get('right_column', ''), actual_columns)
            if right_column:
                referenced.append(right_column)

    seen = set()
    ordered = []
    for column_name in referenced:
        if column_name and column_name not in seen:
            seen.add(column_name)
            ordered.append(column_name)
    return ordered


try:
    CUSTOM_RULES = _parse_rules_payload(CUSTOM_RULES_JSON)
except Exception as exc:
    raise ValueError(str(exc))

if not isinstance(CUSTOM_RULES, list):
    raise ValueError("custom_rules_json must decode to a list")

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Accuracy Check ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Rules provided: {len(CUSTOM_RULES)}")


# ==================================================
# Read input data
# ==================================================
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"Reading from: {input_table}")
df_input = spark.table(input_table)

total_rows = df_input.count()
print(f"Total rows: {total_rows}")

actual_columns = {c.lower(): c for c in df_input.columns}

df_checks = df_input.select("row_id")

enabled_rules = []
for idx, rule in enumerate(CUSTOM_RULES, start=1):
    normalized = _normalize_rule(rule, idx)
    if normalized and normalized.get('enabled', True):
        enabled_rules.append(normalized)

if not enabled_rules:
    raise ValueError("No enabled custom accuracy rules were provided")

print(f"Enabled rules: {len(enabled_rules)}")

for idx, rule in enumerate(enabled_rules, start=1):
    rule_name = rule.get('name') or f'Rule {idx}'
    blank_when_not_applicable = rule.get('blank_when_not_applicable', True)
    rule_columns = rule.get('columns', []) or []

    resolved_rule_columns = _rule_referenced_columns(rule, actual_columns)
    sql_condition = _rule_sql(rule, actual_columns)
    if not sql_condition:
        print(f"Skipping rule with empty SQL translation: {rule_name}")
        continue

    base_column = sanitize_name(rule_columns[0]) if rule_columns else f"rule_{idx:03d}"
    check_col_name = f"{base_column}_{idx:03d}_accuracy_check"

    print(f"Rule {idx}: {rule_name}")
    print(f"  SQL: {sql_condition}")
    print(f"  Check column: {check_col_name}")

    warning_messages = []
    for condition in rule.get('conditions', []) or []:
        if not isinstance(condition, dict):
            continue
        if (condition.get('right_mode', 'column') or 'column').strip().lower() != 'value':
            continue
        warning = _literal_type_warning(condition.get('right_value', ''), condition.get('right_value_type', 'text'))
        if warning:
            warning_messages.append(warning)

    for warning in warning_messages:
        print(f"  {warning}")

    try:
        rule_expr = F.expr(sql_condition)
    except Exception as exc:
        raise ValueError(f"Failed to parse rule '{rule_name}': {exc}\nSQL: {sql_condition}")

    check_expr = F.when(rule_expr, 1).otherwise(0)
    if blank_when_not_applicable and resolved_rule_columns:
        missing_expr = _missing_expression(resolved_rule_columns)
        if missing_expr is not None:
            check_expr = F.when(missing_expr, None).otherwise(check_expr)

    eval_columns = ['row_id'] + resolved_rule_columns
    df_rule = df_input.select(*eval_columns).withColumn(check_col_name, check_expr)
    # Keep only row_id and check column to avoid duplicate column conflicts during join
    df_rule = df_rule.select("row_id", check_col_name)
    df_checks = df_checks.join(df_rule, "row_id", "left")

    valid_count = df_checks.filter(F.col(check_col_name) == 1).count()
    invalid_count = df_checks.filter(F.col(check_col_name) == 0).count()
    null_count = df_checks.filter(F.col(check_col_name).isNull()).count()

    print(f"  Valid rows: {valid_count}")
    print(f"  Invalid rows: {invalid_count}")
    print(f"  NULL rows: {null_count}")


# Add execution timestamp
df_checks = df_checks.withColumn("execution_timestamp", F.current_timestamp())

# Output table name
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_accuracy"
print(f"Writing to: {output_table}")

df_checks.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_checks.count()} rows to {output_table}")

print(f"\nAccuracy Statistics:")
for idx, rule in enumerate(enabled_rules, start=1):
    rule_name = rule.get('name') or f'Rule {idx}'
    rule_columns = rule.get('columns', []) or []
    base_column = sanitize_name(rule_columns[0]) if rule_columns else f"rule_{idx:03d}"
    check_col_name = f"{base_column}_{idx:03d}_accuracy_check"
    valid_count = df_checks.filter(F.col(check_col_name) == 1).count()
    invalid_count = df_checks.filter(F.col(check_col_name) == 0).count()
    null_count = df_checks.filter(F.col(check_col_name).isNull()).count()
    print(f"  {rule_name}: valid={valid_count}, invalid={invalid_count}, null={null_count}")